# Camada Silver: chamados_hora

Trata a tabela `bronze.chamados_hora` e grava `silver.tb_chamados_hora`. Os três horários de cada chamado (abertura, início e finalização) chegam como texto em formatos variados. O notebook extrai deles a data, a hora e o minuto, corrige datas fora do período válido e ajusta os chamados que atravessam a meia-noite. As análises que embasam esses tratamentos estão em `Bronze_Analysis_chamados_hora`.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import date

# criação dos nomes/caminhos dos schemas de cada camada
catalog = "vcredit_medallion"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

## Funções auxiliares

`normalizar_inteiro` mantém só os dígitos de um texto e converte para inteiro. `extrair_datahora` lê textos como "12/07/2025 às 09:23:50" e devolve a data, a hora e o minuto, ignorando o que estiver entre a data e a hora, o que evita o problema de encoding do "à". Data impossível, hora fora de 0 a 23 e minuto fora de 0 a 59 viram nulos.

O intervalo válido vai de 01/01/2025 até o dia da execução. Datas posteriores ao dia da execução são tratadas como erro de digitação. `data_no_intervalo` e `esta_fora` testam se uma data está nesse intervalo, e `VIZINHOS` define, para cada evento, de quais outros eventos a data pode ser emprestada quando estiver fora dele. `minutos_do_dia` converte hora e minuto em minutos desde a meia-noite. `checar` interrompe o notebook se alguma linha violar uma regra de validação.

In [0]:
def normalizar_inteiro(col):
    """
    Remove tudo que não for dígito (0-9) e converte para inteiro (bigint).
    - texto sem nenhum dígito (ex.: 'NULL', '', 'abc') vira null
    - mais de 18 dígitos vira null (evita estouro do bigint)
    Aceita nome de coluna (str) ou Column.
    """
    c = F.col(col) if isinstance(col, str) else col
    digitos = F.regexp_replace(c.cast("string"), r"[^0-9]", "")
    return (
        F.when((digitos == "") | (F.length(digitos) > 18), F.lit(None).cast("long"))
         .otherwise(digitos.cast("long"))
    )

In [0]:
# dd/mm/aaaa  <qualquer coisa que não seja dígito>  hh:mm
# o \D+ cobre " às ", " �s ", " " etc. — não depende do encoding do "à"
PADRAO_DATAHORA = r"(\d{1,2})/(\d{1,2})/(\d{4})\D+(\d{1,2}):(\d{2})"

def extrair_datahora(col, prefixo):
    """
    De strings tipo '12/07/2025 às 09:23:50' extrai:
      {prefixo}_data   -> date (yyyy-MM-dd); data impossível (ex.: 31/02) vira null
      {prefixo}_hora   -> int 0-23; fora da faixa vira null
      {prefixo}_minuto -> int 0-59; fora da faixa vira null
    Retorna um dict para usar com df.withColumns(). Se a string não bater com o padrão, tudo vem null.
    """
    c = F.col(col) if isinstance(col, str) else col
    s = c.cast("string")
    dia, mes, ano, hora, minuto = [F.regexp_extract(s, PADRAO_DATAHORA, i) for i in range(1, 6)]

    # sem match, regexp_extract devolve "" -> texto "--" -> try_to_timestamp devolve null
    data = F.to_date(F.try_to_timestamp(F.concat_ws("-", ano, mes, dia), F.lit("yyyy-M-d")))
    h = normalizar_inteiro(hora)
    m = normalizar_inteiro(minuto)

    return {
        f"{prefixo}_data":   data,
        f"{prefixo}_hora":   F.when(h.between(0, 23), h).cast("int"),
        f"{prefixo}_minuto": F.when(m.between(0, 59), m).cast("int"),
    }

In [0]:
# ---------- limites válidos para as datas ----------
DATA_MINIMA = date(2025, 1, 1)
DATA_MAXIMA = date.today()                     # dia em que o notebook roda

# ordem de preferência dos vizinhos para cada evento: o evento mais próximo primeiro
VIZINHOS = {
    "abertura": ["inicio", "fim"],
    "inicio":   ["abertura", "fim"],
    "fim":      ["inicio", "abertura"],
}

def data_no_intervalo(prefixo):
    """A data do evento, se estiver dentro dos limites; senão null."""
    d = F.col(f"{prefixo}_data")
    return F.when(d.between(F.lit(DATA_MINIMA), F.lit(DATA_MAXIMA)), d)

def esta_fora(prefixo):
    """True quando a data existe mas está fora dos limites (null não conta como fora)."""
    d = F.col(f"{prefixo}_data")
    return d.isNotNull() & ~d.between(F.lit(DATA_MINIMA), F.lit(DATA_MAXIMA))

def minutos_do_dia(p):
    return F.col(f"{p}_hora") * 60 + F.col(f"{p}_minuto")


def checar(regra, df_falhas):
    """Para o notebook se alguma linha violar a regra."""
    n = df_falhas.count()
    assert n == 0, f"FALHA na validação '{regra}': {n} linha(s)"
    print(f"OK: {regra}")

## 1. Leitura da Bronze

A Bronze é gravada em append, então cada execução da ingestão acrescenta uma nova carga completa. Para cada `id_chamado`, ficam só as linhas da carga mais recente. Linhas repetidas dentro da mesma carga são mantidas, para que a validação da seção 5 as detecte.

In [0]:
# para cada id_chamado, só as linhas da carga mais recente (a Bronze é append)
w_carga = Window.partitionBy("ID_Chamado")

df_bronze_chamados_hora = (
    spark.table(f"{bronze_schema}.chamados_hora")
    .withColumn("_ultima_carga", F.max("ingestion_datetime").over(w_carga))
    .filter(F.col("ingestion_datetime") == F.col("_ultima_carga"))
    .drop("_ultima_carga")
)

## 2. Extração de data, hora e minuto

As colunas de id são renomeadas para minúsculas, e cada uma das três colunas de horário dá origem a três colunas: data, hora e minuto. As colunas originais e o `ingestion_datetime` são descartados.

In [0]:
df_extraido = (
    df_bronze_chamados_hora
    .withColumnRenamed("ID_Chamado", "id_chamado")
    .withColumnRenamed("ID_Cliente", "id_cliente")
    .withColumns(extrair_datahora("Hora_Abertura_Chamado", "abertura"))
    .withColumns(extrair_datahora("Hora_Inicio_Atendimento", "inicio"))
    .withColumns(extrair_datahora("Hora_Finalizacao_Atendimento", "fim"))
    .drop("Hora_Abertura_Chamado", "Hora_Inicio_Atendimento", "Hora_Finalizacao_Atendimento", "ingestion_datetime")
)

## 3. Datas fora do intervalo

Uma data fora do intervalo válido é trocada pela data do evento vizinho mais próximo que esteja dentro dele. Se nenhum vizinho for válido, ela fica nula. A célula mostra quantas datas de cada evento estavam fora do intervalo e quantas foram corrigidas.

In [0]:
# data substituta de cada evento: a primeira data válida entre os vizinhos
substituta = {p: F.coalesce(*[data_no_intervalo(v) for v in viz]) for p, viz in VIZINHOS.items()}

contagem = df_extraido.agg(*[
    expr
    for p in VIZINHOS
    for expr in (
        F.sum(F.when(esta_fora(p), 1).otherwise(0)).alias(f"{p}_fora"),
        F.sum(F.when(esta_fora(p) & substituta[p].isNotNull(), 1).otherwise(0)).alias(f"{p}_corrigida"),
    )
]).first()

print(f"Intervalo válido: {DATA_MINIMA} até {DATA_MAXIMA}")
for p in VIZINHOS:
    fora = contagem[f"{p}_fora"] or 0
    corrigida = contagem[f"{p}_corrigida"] or 0
    print(f"  {p}: {fora} fora do intervalo -> {corrigida} corrigidas por outra data da linha, "
          f"{fora - corrigida} viraram null")

df_silver_chamados_hora = df_extraido.withColumns({
    f"{p}_data": F.when(esta_fora(p), substituta[p]).otherwise(F.col(f"{p}_data"))
    for p in VIZINHOS
})

## 4. Virada de meia-noite

Quando um evento tem a mesma data do anterior, mas um horário menor, ele aconteceu no dia seguinte. Nesses casos, a data do evento posterior avança um dia.

In [0]:
# virada de meia-noite: mesmo dia do evento anterior, mas horário menor -> foi no dia seguinte
for anterior, posterior in [("abertura", "inicio"), ("inicio", "fim")]:
    virou_o_dia = (
        (F.col(f"{posterior}_data") == F.col(f"{anterior}_data"))
        & (minutos_do_dia(posterior) < minutos_do_dia(anterior))
    )
    df_silver_chamados_hora = df_silver_chamados_hora.withColumn(
        f"{posterior}_data",
        F.when(virou_o_dia, F.date_add(f"{posterior}_data", 1)).otherwise(F.col(f"{posterior}_data")),
    )

display(df_silver_chamados_hora)

## 5. Validações

A Gold faz join da fato com esta tabela pelo `id_chamado`. Um id nulo ou repetido duplicaria ou perderia chamados na fato, então o notebook para se isso acontecer.

In [0]:
checar("chamados_hora: id_chamado preenchido",
       df_silver_chamados_hora.filter(F.col("id_chamado").isNull()))
checar("chamados_hora: id_chamado único",
       df_silver_chamados_hora.groupBy("id_chamado").count().filter("count > 1"))

## 6. Gravação da Silver

Só chega aqui se todas as validações passaram.

In [0]:
(df_silver_chamados_hora.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{silver_schema}.tb_chamados_hora"))
print(f"\nGravado em {silver_schema}.tb_chamados_hora")